#LangSmith와 Ragas를 활용한 오프라인 평가 구성


* Ragas는 RAG 파이프라인의 품질을 평가하기 위한 도구입니다.
* Ragas는 내부적으로 LLM을 사용하여 답변 품질, 관련성 등을 평가합니다.







##환경 설정 및 패키지 설치

In [ ]:
import os
from google.colab  import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "ragas-project"
os.environ["OPEN_API_KEY"] = userdata.get("OPENAI_API_KEY")


In [ ]:
# 핵심 패키지들
!pip install langchain langchain-openai langchain-community langchain-text-splitters

In [ ]:
# LangChain v0.3+ 호환 버전
!pip install langchain-openai>=0.2.0

In [ ]:
# GitPython은 Python 코드로 Git 저장소를 다룰 수 있게 해주는 라이브러리입니다.
!pip install GitPython

Ragas 패키지 설치

In [ ]:
!pip install ragas==0.3.7 datasets -U

In [ ]:
# Ragas 버전 확인
!pip show ragas

In [ ]:
# 추론 함수에 사용
!pip install chromadb

In [ ]:
!pip show chromadb

##Ragas를 활용한 합성 테스트 (Synthetic data) 데이터 생성

*  LLM, 시뮬레이션 등을 통해 인공적으로 생성된 데이터를 합성 데이터(Synthetic data) 라고 합니다.
* 합성 테스트 데이터 목적:
  * RAG 모델 또는 일반 LLM이 특정 문맥, 질문 유형, 시나리오에서 어떻게 답변하는지 평가하기 위해서 활용됩니다.
* 합성 테스트는 LLM 입력(Question + Contexts) → LLM 출력 → Reference와 비교 → 평가의 구조로 이루어집니다.

* 합성 테스트의 구성 요소:
  * Question (user_input) : 테스트 질문. 모델에게 입력되는 실제 쿼리입니다.

  * Contexts (retrieved_contexts) :질문에 관련된 문맥 정보. LLM이 답변 생성 시 참고합니다.
  * LLM : 질문과 문맥을 입력으로 받아 답변(response) 생성.
  * LLM Output (response) : 모델이 실제로 생성한 답변.
  * Reference Answer (reference) : 합성 테스트에서 정의된 정답 기준.
  * LLM Output과 비교하여 정확도, faithfulness, relevancy 등을 평가합니다.


###Ragas 평가 대상 문서 로드 및 전처리

* Ragas 평가 대상 문서로 "LangChain의 공식 문서"를 사용하겠습니다.
* https://github.com/langchain-ai/langchain


In [ ]:

from langchain_community.document_loaders import GitLoader


document_loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=lambda file_path: file_path.endswith(".md"),
)

documents = document_loader.load()

print(f"Document 갯수 : {len(documents)}");  # 42



Document 갯수 : 42


In [ ]:
from langchain_core.documents import Document


for i, doc in enumerate(documents):
    print(f"Document {i}: type={type(doc)}")
    if isinstance(doc, Document):
        print(f"  page_content type={type(doc.page_content)}, length={len(doc.page_content)}")
    else:
        print(f"  content={doc}")

빈 Document를 제거합니다.

In [ ]:
documents = [doc for doc in documents if len(doc.page_content.strip()) > 0]

In [ ]:
print(f"Document 갯수 : {len(documents)}");  # 41

Document 갯수 : 41


###메타데이터 설정
* Ragas가 사용하는 메타데이터인 "filename"을 설정합니다.
* filename 메타데이터는 테스트 데이터셋 생성 과정에서 동일한 문서에 속한 청크를 식별하는 데 사용됩니다.

In [ ]:
for document in documents:
    document.metadata['filename'] = document.metadata['source']

In [ ]:
for document in documents:
    print(document.metadata)

###합성 테스트 데이터셋 생성
Ragas의 기능으로 합성 테스트 데이터를 생성합니다.
  * LLM과 임베딩 모델이 필요합니다.

In [ ]:
from ragas.llms.base import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from openai import OpenAI
from ragas.embeddings import OpenAIEmbeddings

# LangchainLLMWrapper : LangChain 모델을 RAGAS에서 쓸 수 있도록 래핑(wrapping) 하는 도우미 클래스
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=os.environ["OPEN_API_KEY"])
)

openai_client = OpenAI(api_key=os.environ["OPEN_API_KEY"])
generator_embeddings = OpenAIEmbeddings(client=openai_client)


In [ ]:
# 문자열 유사도(String similarity)를 매우 빠르게 계산하는 라이브러리입니다.
# Ragas의 TestsetGenerator는 내부적으로 문장 유사도 계산을 위해 rapidfuzz 를 사용합니다.

!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.8 MB/s eta 0:00:00


* Ragas 라이브러리를 이용해 RAG 시스템용 테스트셋을 자동으로 생성하려는 예시
  * TestsetGenerator : 합성 테스트셋(질의-답변 쌍 등)을 자동 생성하는 도구
    * llm : BaseLanguageMode : 테스트셋 생성에 필요한 모든 작업(질문 생성, 컨텍스트 비평, 정답 생성 등)을 담당하는 주 언어 모델입니다.
    - user_input : 사용자 질문
    - reference_contexts : 모델이 답변을 생성할 때 참고해야 하는 문서나 컨텍스트 목록.
    - reference : 정답/참조 답변(reference answer)
    - synthesizer_name : 질문 유형과 답변 생성 전략

In [ ]:
from ragas.testset import TestsetGenerator
from ragas.evaluation import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.embeddings import OpenAIEmbeddings
from datasets import load_dataset
from langchain_core.documents import Document

try:
  generator = TestsetGenerator(
      llm=generator_llm,
      embedding_model=generator_embeddings,
  )

  # 질문 유형 분포
  query_distribution = {
    "single_hop_specific_query_synthesizer": 0.5,  # 단일 문서를 기반으로 한 질문을 50%
    "multi_hop_specific_query_synthesizer": 0.25,  # 여러 문서를 기반으로 한 구체적 질문을 25%
    "multi_hop_abstract_query_synthesizer": 0.25   # 여러 문서를 기반으로 한 추상적 질문을 25%
  }

  # LangChain 형식의 Document 리스트를 입력으로 받아 RAG 평가용 "질문·정답·evidence" 테스트 세트를 자동 생성하는 함수
  # query_distribution 기본값 때문에 자동으로 최소 6개의 질문이 생성됩니다.
  testset = generator.generate_with_langchain_docs(
      documents,
      testset_size=6,            # 6개의 질문 생성
      #query_distribution=query_distribution
  )

except Exception as e:
  print(f"초기화 중 오류 발생: {e}")

Applying HeadlinesExtractor:   0%|          | 0/11 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/41 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/21 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
testset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What are the core development principles for L...,[# Global Development Guidelines for LangChain...,The core development principles for LangChain ...,single_hop_specific_query_synthesizer
1,How should API documentation be handled accord...,[Conventional Commits format for PR titles:** ...,API documentation should include comprehensive...,single_hop_specific_query_synthesizer
2,What are the key features of LangChain as an o...,[<1-hop>\n\n# 🦜️🔗 LangChain\n\n[![PyPI - Versi...,LangChain is an open-source project designed t...,multi_hop_abstract_query_synthesizer
3,What are the key features of LangChain as an o...,[<1-hop>\n\n# 🦜️🔗 LangChain\n\n[![PyPI - Versi...,LangChain is an open-source project designed t...,multi_hop_abstract_query_synthesizer
4,How can I report security vulnerabilities rela...,[<1-hop>\n\nReporting OSS Vulnerabilities\n\nL...,To report security vulnerabilities related to ...,multi_hop_specific_query_synthesizer
5,What are the guidelines for implementing multi...,[<1-hop>\n\nadd multi-tenant support` - `!fix(...,The guidelines for implementing multi-tenant s...,multi_hop_specific_query_synthesizer


##LangSmith의 Dataset 생성
* 오프라인 평가에 사용할 데이터 셋은 저장 관리가 중요합니다.
* LangSmith에는 평가용 데이터셋을 관리하는 기능을 제공합니다.


###데이터셋 생성
LangSmith에 데이터셋을 관리하는 'Dataset'이라는 객체를 생성합니다.

In [ ]:

# 데이터셋 생성 함수

def create_dataset(client, dataset_name, description=None):

    """
    client : LangSmith 서버에 연결할 클라이언트 인스턴스
    dataset_name : 데이터셋 이름
    """
    # 기존 데이터셋 확인
    datasets = client.list_datasets(dataset_name=dataset_name)
    for dataset in datasets:
        if dataset.name == dataset_name:
            return dataset

    # 새 데이터셋 생성
    new_dataset = client.create_dataset(dataset_name=dataset_name, description=description)
    return new_dataset


In [ ]:
# 데이터 셋 생성

from langsmith import Client

client = Client()

DATASET_NAME = "mzc"
dataset = create_dataset(client, DATASET_NAME)

In [ ]:
# 데이터 셋 아이디 확인
print(dataset.id)

9c5a9743-e52c-43b1-9094-17fd3142ffcc


###데이터셋에 Example 추가
* Ragas에서 생성한 합성 테스트 데이터셋을 LangSmith의 Dataset에 업로드합니다.



In [ ]:
from ragas.testset.synthesizers.testset_schema import TestsetSample


# Ragas 0.3.7 기준, 실제 샘플 목록은 testset.samples 속성에 들어있습니다.
# sample : TestsetSample
for sample in testset.samples:

    # SingleTurnSample : 실제 질문·문맥·정답 데이터가 들어있는 객체입니다.
    # user_input : 실제 모델에게 입력될 쿼리
    # retrieved_contexts : 질문과 관련된 문맥들
    # reference : 정답 또는 기대 출력
    inner = sample.eval_sample

    question = inner.user_input
    contexts = inner.retrieved_contexts
    answer = inner.reference

    # LangSmith에 하나의 Example 업로드
    client.create_example(
        dataset_id=dataset.id,                                           # 업로드할 대상 Dataset의 ID
        inputs={"question": question, "contexts": contexts},             # Ragas가 합성 테스트 데이터를 만들 때 LLM에 제공한 것과 동일한 정보, LLM에 실제 입력되는 질문과 문맥
        outputs={"answer": answer}                                       # 합성 테스트에서 모델이 맞춰야 하는 정답 기준
    )

print(f"LangSmith Dataset '{DATASET_NAME}' has been uploaded successfully!!")


LangSmith Dataset 'mzc' has been uploaded successfully!!


##LangSmith와 Ragas를 활용한 오프라인 평가 구현
Ragas 평가 매트릭
- 검색 평가 매트릭 :
  - 평가 대상 : RAG에서 검색된 문서 / Context
  - Context-precision: 검색한 문서 중 정답 포함 여부
- 생성 평가 매트릭 :
  - 평가 대상 : LLM이 생성한 답변
  - Answer-relevancy(답변 관련성) : 답변과 정답의 의미적 유사도



###평가 매트릭 준비
* RagasMetricEvaluator(커스텀 클래스)를 사용하여  Ragas의 평가 매트릭을 준비합니다.


###커스텀 Evaluator 구현

In [ ]:
from langsmith.schemas import Run, Example                                  # LangSmith에서 하나의 평가 데이터(Example)와 실행 결과(Run)를 나타냄
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper                                  # LangChain 모델을 RAGAS에서 쓸 수 있도록 래핑(wrapping) 하는 도우미 클래스
from langchain_core.embeddings import Embeddings
from langchain_core.language_models import BaseChatModel
from typing import List
from ragas.metrics.base import SingleTurnSample
from ragas.metrics.base import Metric, MetricWithEmbeddings, MetricWithLLM   # RAGAS의 평가 지표 기반 클래스들


# -------------------------
# RagasMetricEvaluator 정의
# -------------------------
class RagasMetricEvaluator:

    def __init__(self, metric: Metric, llm: BaseChatModel, embeddings: Embeddings):
        """
        metric : 평가 지표 객체 (예: context_precision, answer_relevancy ...)
        llm : LLM 모델 객체
        embeddings : 임베딩 모델 객체
        """
        self.metric = metric

        # Ragas 평가 매트릭을 사용하여 평가하려면 LLM과 Embeddings를 Metric에 설정해야 합니다.
        if isinstance(metric, MetricWithLLM):
            metric.llm = LangchainLLMWrapper(llm)
        if isinstance(metric, MetricWithEmbeddings):
            metric.embeddings = LangchainEmbeddingsWrapper(embeddings)


    # LangSmith의 Run + Dataset Example을 입력으로 받아 Ragas Metric 점수를 계산하는 평가기(Evaluator) 입니다.
    async def evaluate(self, run: Run, example: Example) -> dict:
        """
          run: LangSmith에서의 실행 결과 (execution record)
          example: LangSmith에서의 평가 데이터셋의 한 항목
          반환값 : 반드시 숫자/불린 타입이어야 함.
        """

        # list comprehension
        # 각 Document에서 page_content만 꺼내 문자열 리스트로 만듬
        context_strs = [doc.page_content for doc in run.outputs.get('contexts', [])]

        # Ragas 평가 샘플 구성
        # SingleTurnSample: Ragas에서 단일 질문-응답 샘플을 평가할 때 사용하는 데이터 구조
        # user_inupt : 사용자 질문 (example)
        # reference : 정답 / 기준답변 (example)
        # response : LLM 모델이 생성한 답변 (run)
        # retrieved_contexts : RAG에서 검색된 문서, 모델이 답변을 생성할 때 참고해야 하는 컨텍스트
        row = SingleTurnSample(
            user_input=example.inputs.get("question") or example.inputs.get("user_input") or "",
            retrieved_contexts=context_strs or [],
            response=run.outputs.get("answer") or "",
            reference=example.outputs.get("reference") or ""
        )

        # Ragas Metric을 사용하여 해당 row의 평가 점수를 계산한다.
        raw_score = await self.metric.single_turn_ascore(row)

        # raw_score가 coroutine이면(비동기) await, raw_score가 일반 값이면 그대로 사용
        if callable(getattr(raw_score, "__await__", None)):
            score = await raw_score
        else:
            score = raw_score


        # LangSmith에서 요구하는 타입 숫자나 불린으로 변환
        if not isinstance(score, (int, float, bool)):
            try:
                score = float(score)
            except Exception:
                score = 0.0  # 변환 불가 시 기본값

        # LangSmith Metric 포맷
        # {
        #    "key": "ContextPrecision",
        #    "score": 0.87
        # }
        return {"key": type(self.metric).__name__, "score": score}




###추론 함수 구현

* LangChain 기반 RAGAS 평가 파이프라인으로 모델이 실제로 사용자 질문에 답변을 생성하는 함수입니다.

* Ragas는 추론 함수에서 생성한 answer, contexts를 사용하여 평가 메트릭을 계산합니다.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import answer_relevancy, context_precision
from langsmith.evaluation.evaluator import DynamicRunEvaluator
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.environ["OPEN_API_KEY"])

db = Chroma.from_documents(documents, embeddings)

prompt = ChatPromptTemplate.from_template('''
  다음 문맥만을 고려해 질문에 답하세요.
  문맥: """{context}"""
  질문: {question}
''')

model = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.environ["OPEN_API_KEY"])

retriever = db.as_retriever()

chain = RunnableParallel(
    {
      "question": RunnablePassthrough(),
      "context": retriever
    }
).assign(
    answer=prompt | model | StrOutputParser()
)



In [ ]:

# -------------------------------
# 추론 함수
# -------------------------------

from typing import Any, Dict
from langsmith.schemas import Run


def predict(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """
    inputs: {"question": "..."}
    """
    question = inputs.get("question", "")
    if not question:
        return {"contexts": [], "answer": ""}


    output = chain.invoke(question)

    contexts = output.get("contexts") or output.get("context") or []
    answer = output.get("answer", "")

    return {
        "contexts": contexts,  # RAG에서 검색된 문서
        "answer": answer,      # LLM 모델 답변
    }


In [ ]:

import uuid
import datetime
from langsmith.schemas import Run, Example


# LangSmith에서 합성 테스트 데이터셋에 대해 커스텀 평가(custom evaluation) 실행한 경우 기록하려면 Run 객체를 반드시 생성해야 합니다.
def create_run_from_example(example: Example) -> Run:
    """
    example: LangSmith에서의 평가 데이터셋의 한 항목
    Run : LangSmith에서의 실행 결과 (execution record)

    """

    # 추론 함수 실행 : 단일 Example에 대한 LLM 모델 실행
    output = predict(example.inputs)

    # Run 객체는 LangSmith에서 하나의 실행 단위(Execution)를 나타내는 핵심 데이터 구조
    # LangChain, LLM, Tool, Retriever 등이 실행될 때마다 LangSmith는 그 실행 과정 전체를 하나의 Run으로 기록니다.(실행 로그 객체 생성)
    run = Run(
        id=str(uuid.uuid4()),         # 고유 ID
        name="test_run",              # Run 이름
        start_time = datetime.datetime.now(datetime.timezone.utc), #특정 Run이 언제 시작되었는지를 나타내는 타임스탬프
        run_type="manual",            # LangSmith가 자동으로 생성한 Run이 아니라, 개발자가 직접 생성한 Run으로, 임의의 커스텀 평가 실행할때 사용
        trace_id=str(uuid.uuid4()),   # Trace ID : 하나의 전체 실행 흐름(Trace)을 식별하는 ID , 여러 Run을 하나의 흐름으로 묶을 수 있습니다.
        inputs=example.inputs,        # 어떤 입력을 받았는지
        outputs=output                # 어떤 출력을 생성했는지 (LLM 모델의 출력)
    )

    return run


##오프라인 평가 구현 및 실행

In [ ]:

# --------------------------------------------------------------
# LangSmith Dataset에서 Example 불러오기
# --------------------------------------------------------------

from langsmith import Client
client = Client(api_key=os.environ['LANGSMITH_API_KEY'])


def get_examples_from_dataset(dataset_name: str):
    examples = client.list_examples(dataset_name=dataset_name)
    return list(examples)


In [ ]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.environ["OPEN_API_KEY"])
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.environ["OPEN_API_KEY"])


async def evaluate_dataset(dataset_name: str):

    client = Client()
    examples = get_examples_from_dataset(dataset_name)

    # ----------------- Ragas 평가 메트릭 목록 --------------------------------------------
    metrics = [context_precision, answer_relevancy]
    evaluators = [RagasMetricEvaluator(metric, llm, embeddings) for metric in metrics]

    print(f'평가 매트릭의 수 : {len(evaluators)}')

    all_results = []

    for example in examples:

        # ---------------------------------------------------
        # 1) 모델 실행 결과 → Run 객체 생성
        # ---------------------------------------------------
        run = create_run_from_example(example)

        # 1-1) LangSmith에 Run 기록 (Example과 연결)
        client.create_run(
            run=run,
            example_id=example.id
        )

        # ---------------------------------------------------
        # 2) 각 Metric 별 평가 실행
        # ---------------------------------------------------
        scores = {}

        for evaluator in evaluators:
            score_dict = await evaluator.evaluate(run, example)
            metric_name = score_dict["key"]
            metric_value = float(score_dict["score"])

            scores[metric_name] = metric_value

            # -------------------------------------------------------
            # 3) LangSmith에 Evaluation 결과 기록
            # -------------------------------------------------------
            client.log_evaluation(
                run_id=run.id,
                evaluator_name=metric_name,  # 평가 지표 이름
                score=metric_value,
                comment=f"{metric_name} evaluation score"
            )

        # 내부 반환용 리스트
        all_results.append({
            "example_id": example.id,
            "scores": scores
        })

    return all_results


In [ ]:

# -------------------------------
# 평가 실행
# -------------------------------
dataset_name = "mzc"  # LangSmith dataset UUID

async def main():
    results = await evaluate_dataset(dataset_name)
    for r in results:
        print(r)

await main()

/tmp/ipython-input-4070374161.py:26: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  metric.llm = LangchainLLMWrapper(llm)
/tmp/ipython-input-4070374161.py:28: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  metric.embeddings = LangchainEmbeddingsWrapper(embeddings)


평가 매트릭의 수 : 2
{'example_id': UUID('6bd1e4e7-fd40-4b9f-b39e-496cd6b476d1'), 'scores': {'ContextPrecision': 0.0, 'AnswerRelevancy': 0.0}}
{'example_id': UUID('706c5716-4b3e-487a-aeae-733dd74afd7a'), 'scores': {'ContextPrecision': 0.0, 'AnswerRelevancy': 0.8390715960598761}}
{'example_id': UUID('181a679a-f050-4943-bc82-e0aa3f777caf'), 'scores': {'ContextPrecision': 0.0, 'AnswerRelevancy': 0.8045758812218206}}
{'example_id': UUID('95a02c12-bcb4-4d87-bdfe-78eabafb23d4'), 'scores': {'ContextPrecision': 0.0, 'AnswerRelevancy': 0.8095263837280866}}
{'example_id': UUID('76127d13-ef9a-4a35-ae7b-719152d330ea'), 'scores': {'ContextPrecision': 0.3333333333, 'AnswerRelevancy': 0.9135058852064333}}
{'example_id': UUID('de55aec8-7c4c-4f66-bf18-afb79d11ad23'), 'scores': {'ContextPrecision': 0.0, 'AnswerRelevancy': 0.999999476567397}}
